# Shipment Volume Forecasting & Truck Planning Model

Dataset: DataCo Smart Supply Chain (Kaggle). Goal: study daily shipment volume, forecast it with a 7-day moving average, and flag days that need an adhoc truck or a cancellation. Truck capacity is an assumption.

## 1. Load data and prepare one row per order

In [2]:
import pandas as pd, numpy as np, zipfile, os
# Upload the Kaggle zip (or the CSV) to Colab first, then run this cell
for f in os.listdir("."):
    if f.endswith(".zip"):
        zipfile.ZipFile(f).extractall()
df = pd.read_csv("DataCoSupplyChainDataset.csv", encoding="latin-1")

real, sched = "Days for shipping (real)", "Days for shipment (scheduled)"
late, region, mode = "Late_delivery_risk", "Order Region", "Shipping Mode"
odate, qty, sales = "order date (DateOrders)", "Order Item Quantity", "Sales"

# one row per order (raw file has one row per order item)
o = df.groupby("Order Id").agg(
    date=(odate, "first"), real=(real, "max"), sched=(sched, "max"),
    late=(late, "max"), mode=(mode, "first"), region=(region, "first"),
    status=("Delivery Status", "first"), qty=(qty, "sum"), sales=(sales, "sum")
).reset_index()
o["date"] = pd.to_datetime(o["date"])
o["delay_days"] = o["real"] - o["sched"]
print("Orders:", len(o), "| Date range:", o["date"].min().date(), "to", o["date"].max().date())

Orders: 65752 | Date range: 2015-01-01 to 2018-01-31


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Volume patterns and forecast

In [3]:
CAP = 200   # ASSUMPTION: units per truck (not from the dataset)

daily = o.groupby(o.date.dt.normalize()).agg(orders=("Order Id","count"), units=("qty","sum"))
print("Days of data:", len(daily), "| Avg daily orders:", round(daily.orders.mean(),1), "| Peak:", daily.orders.max())

print("\nAvg orders by weekday:")
print(daily.orders.groupby(daily.index.day_name()).mean().round(1))

# 7-day moving average forecast (on units), shifted by one day so it uses only past data
daily["forecast_units"] = daily.units.rolling(7).mean().shift(1)
d = daily.dropna().copy()
mape = ((d.units - d.forecast_units).abs() / d.units).mean() * 100
print("\n7-day moving average forecast error (MAPE): %", round(mape, 1))

Days of data: 1127 | Avg daily orders: 58.3 | Peak: 69

Avg orders by weekday:
date
Friday       58.4
Monday       58.2
Saturday     58.6
Sunday       58.2
Thursday     58.4
Tuesday      58.4
Wednesday    58.4
Name: orders, dtype: float64

7-day moving average forecast error (MAPE): % 9.3


## 3. Truck planning (assumed capacity: 200 units per truck)

In [4]:
# Truck planning: trucks needed from forecast vs a baseline plan
d["trucks_needed"] = np.ceil(d.forecast_units / CAP)
planned = int(np.ceil(d.units.mean() / CAP))
d["trucks_planned"] = planned
d["action"] = np.where(d.trucks_needed > planned, "ADD adhoc truck",
              np.where(d.trucks_needed < planned, "CANCEL extra truck", "OK"))
print("Planned trucks per day (baseline):", planned)
print(d.action.value_counts())
d.reset_index().rename(columns={"date":"day"}).to_excel("truck_planning.xlsx", index=False)
print("Saved truck_planning.xlsx")

Planned trucks per day (baseline): 2
action
OK                    958
CANCEL extra truck    117
ADD adhoc truck        45
Name: count, dtype: int64
Saved truck_planning.xlsx
